# 🌐 MonoSplat — Gaussian Splat Training on Colab GPU

## Before you run this notebook

This notebook only handles **Gaussian Splat training**. FFmpeg and COLMAP already ran on your desktop.

You need to have already:
- ✅ Started the MonoSplat server: `uvicorn src.pipeline.server:app --reload --port 8000`
- ✅ Uploaded a video at `http://localhost:8000`
- ✅ Watched the progress bar complete the **Frames** and **COLMAP** stages (it will stall at Train — that is expected)
- ✅ Noted your `job_id` from `models/registry.json`

---

## Step 0 — Create your zip on the desktop

**Windows (PowerShell) — replace `<job_id>` with your actual job_id:**
```powershell
Compress-Archive -Path work\<job_id>\frames, work\<job_id>\colmap, config -DestinationPath monosplat_job.zip
```

**Mac / Linux:**
```bash
zip -r monosplat_job.zip work/<job_id>/frames work/<job_id>/colmap config/
```

The zip must contain these three things:
```
monosplat_job.zip
├── work/
│   └── <job_id>/
│       ├── frames/          ← PNG frames extracted by FFmpeg
│       └── colmap/
│           └── sparse_text/ ← cameras.txt, images.txt, points3D.txt
└── config/
    └── config.yaml
```

Then come back here and run cells **top to bottom**.

---

## After training — what to do on your desktop

Cell 7 will download the `.splat` and `.ply` files. Then on your desktop:
1. Copy the files to `work/<job_id>/models/gaussian/`
2. Open `models/registry.json`, find your job and update:
   ```json
   "status": "ready",
   "splat_path": "work/<job_id>/models/gaussian/<job_id>.splat",
   "ply_path":   "work/<job_id>/models/gaussian/<job_id>.ply"
   ```
3. Open `http://localhost:8000/viewer/<job_id>` — your scene is live in the browser.

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────
# You must see a GPU listed. If not: Runtime → Change runtime type → T4 GPU.

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅  GPU detected:')
    # Print just the first useful lines
    for line in result.stdout.split('\n')[:12]:
        print(line)
else:
    print('❌  No GPU found!')
    print('   → Go to Runtime → Change runtime type → T4 GPU, then re-run all cells.')
    raise SystemExit('No GPU. Change runtime type first.')

In [2]:
# ── Cell 2: Install dependencies ────────────────────────────────────
# PyTorch is pre-installed on Colab. We only need a few extras.
# Takes ~30 seconds.

!pip install -q pillow pyyaml plyfile tqdm numpy

import torch
print(f'✅  PyTorch {torch.__version__}')
print(f'   CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU            : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌  CUDA not available — check your runtime type.')
    raise SystemExit('CUDA not available. Change runtime type to T4 GPU.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.9 MB/s eta 0:00:00
✅  PyTorch 2.10.0+cu128
   CUDA available : True
   GPU            : Tesla T4
   VRAM           : 15.6 GB


In [3]:
# ── Cell 3: Upload monosplat_job.zip ────────────────────────────────
# When the file picker opens, select your monosplat_job.zip.
# The zip must contain: work/<job_id>/frames/  work/<job_id>/colmap/  config/
#
# See the instructions at the top of this notebook for how to create the zip.

from google.colab import files
import zipfile, os, glob

print('📂  Select your monosplat_job.zip when the picker opens...')
print('   (This may take a moment to upload depending on file size)')
print()
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded. Please upload monosplat_job.zip and re-run this cell.')

zip_name = list(uploaded.keys())[0]
zip_size = os.path.getsize(zip_name) / 1e6
print(f'\n   Uploaded : {zip_name}  ({zip_size:.1f} MB)')

print('   Extracting to /content/monosplat ...')
os.makedirs('/content/monosplat', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/monosplat')

os.chdir('/content/monosplat')
print(f'   Working directory: {os.getcwd()}')

# Auto-detect job_id from the work/ folder
work_dirs = [d for d in glob.glob('work/*') if os.path.isdir(d)]
if not work_dirs:
    raise RuntimeError(
        '\n❌  Could not find work/<job_id>/ in the zip.'
        '\n   Make sure you included: work/<job_id>/frames  work/<job_id>/colmap  config/'
        '\n   Re-create the zip and try again.'
    )

JOB_ID      = os.path.basename(work_dirs[0])
FRAMES_DIR  = f'work/{JOB_ID}/frames'
COLMAP_DIR  = f'work/{JOB_ID}/colmap/sparse_text'
OUTPUT_DIR  = f'work/{JOB_ID}/models/gaussian'
CKPT_DIR    = f'work/{JOB_ID}/models/checkpoints'
CONFIG_PATH = 'config/config.yaml'

print(f'\n✅  Job ID  : {JOB_ID}')
print(f'   Frames  : {FRAMES_DIR}')
print(f'   COLMAP  : {COLMAP_DIR}')
print(f'   Output  : {OUTPUT_DIR}')
print(f'\n→  Run Cell 4 to verify COLMAP files before training.')

📂  Select your monosplat_job.zip when the picker opens...
   (This may take a moment to upload depending on file size)



Saving 4d1fe58403ff_for_colab.zip to 4d1fe58403ff_for_colab.zip

   Uploaded : 4d1fe58403ff_for_colab.zip  (577.9 MB)
   Extracting to /content/monosplat ...
   Working directory: /content/monosplat

✅  Job ID  : 4d1fe58403ff
   Frames  : work/4d1fe58403ff/frames
   COLMAP  : work/4d1fe58403ff/colmap/sparse_text
   Output  : work/4d1fe58403ff/models/gaussian

→  Run Cell 4 to verify COLMAP files before training.


In [ ]:
# ── Cell 4: Bootstrap src/ from GitHub ──────────────────────────────
# Forces a clean re-clone of src/ and scripts/ so you always get the latest fixes.
# If src/ is already present from a previous run, it is removed first.
import os, subprocess, shutil, sys

src_dir     = os.path.join(os.getcwd(), "src")
scripts_dir = os.path.join(os.getcwd(), "scripts")

# Always clean and re-fetch to pick up latest GitHub fixes
for d in ["/tmp/monosplat_src", src_dir, scripts_dir]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"🗑️  Removed old {d}")

print("📦  Fetching src/ and scripts/ from GitHub...")
r1 = subprocess.run([
    "git", "clone", "--filter=blob:none", "--sparse",
    "https://github.com/Somaskandan931/monosplat.git", "/tmp/monosplat_src"
], capture_output=True, text=True)

r2 = subprocess.run([
    "git", "-C", "/tmp/monosplat_src", "sparse-checkout", "set", "src", "scripts"
], capture_output=True, text=True)

if r1.returncode != 0 or not os.path.isdir("/tmp/monosplat_src/src"):
    print("stderr:", r1.stderr)
    raise RuntimeError("❌  Clone failed. Is https://github.com/Somaskandan931/monosplat public?")

shutil.copytree("/tmp/monosplat_src/src", src_dir)
shutil.copytree("/tmp/monosplat_src/scripts", scripts_dir, dirs_exist_ok=True)
print("✅  src/ and scripts/ fetched from GitHub.")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.reconstruction.gaussian_model import GaussianModel
print("✅  src/ importable — ready for training.")


In [39]:
# ── Cell 5: Verify COLMAP output ────────────────────────────────────
# These three files must exist and have data.
# If any are missing: COLMAP did not complete on your desktop.
# Go back to localhost:8000 and check the COLMAP stage.

import os, glob

print(f'Checking COLMAP output: {COLMAP_DIR}\n')
all_ok = True

for fname in ['cameras.txt', 'images.txt', 'points3D.txt']:
    path = os.path.join(COLMAP_DIR, fname)
    if os.path.exists(path):
        data_lines = [l for l in open(path) if not l.startswith('#') and l.strip()]
        print(f'  ✅  {fname:<18} {len(data_lines):>5} data lines')
    else:
        print(f'  ❌  {fname:<18} MISSING')
        all_ok = False

frames = (glob.glob(os.path.join(FRAMES_DIR, '*.png')) +
          glob.glob(os.path.join(FRAMES_DIR, '*.jpg')))
print(f'\n  📷  Frames : {len(frames)} files in {FRAMES_DIR}')

if not all_ok:
    print('\n❌  COLMAP did not finish. Do NOT run training yet.')
    print('   Check your desktop upload portal — COLMAP must reach 100%.')
    print('   Common causes: not enough image overlap, motion blur, poor lighting.')
elif len(frames) == 0:
    print('\n❌  No frames found. Check your zip — frames/ folder may be empty or missing.')
else:
    print('\n✅  All COLMAP files present and frames found.')
    print('   → Run Cell 5 to configure training, then Cell 6 to start.')

Checking COLMAP output: work/4d1fe58403ff/colmap/sparse_text

  ✅  cameras.txt            1 data lines
  ✅  images.txt           400 data lines
  ✅  points3D.txt        1388 data lines

  📷  Frames : 200 files in work/4d1fe58403ff/frames

✅  All COLMAP files present and frames found.
   → Run Cell 5 to configure training, then Cell 6 to start.


In [40]:
# ── Cell 6: (Optional) Adjust training iterations ───────────────────
# The config is already tuned for free Colab T4 (30k iters, 640x360, 200k Gaussians).
# Uncomment the FAST PREVIEW block below only if you want a quick test run first.
#
# FAST PREVIEW MODE (~15 min): uncomment to enable, then run Cell 6.
# !sed -i 's/iterations: [0-9]*/iterations: 10000/' config/config.yaml
# !sed -i 's/densify_until_iter: [0-9]*/densify_until_iter: 7000/' config/config.yaml

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
torch.cuda.empty_cache()

import yaml
with open(CONFIG_PATH) as f:
    cfg_preview = yaml.safe_load(f)

iters   = cfg_preview['training']['iterations']
est_min = iters // 750

print(f'Training configuration:')
print(f'  Iterations     : {iters:,}')
print(f'  Estimated time : ~{est_min} minutes on T4 GPU')
print(f'  Resolution     : {cfg_preview["viewer"]["window_width"]}x{cfg_preview["viewer"]["window_height"]}')
print(f'  Max Gaussians  : {cfg_preview["renderer"]["max_gaussians"]:,}')
print(f'  SH degree      : {cfg_preview["renderer"]["sh_degree"]}')
print()
print('→  When ready, run Cell 6 to start training.')


Training configuration:
  Iterations     : 30,000
  Estimated time : ~40 minutes on T4 GPU
  Resolution     : 1280x720
  Max Gaussians  : 200,000
  SH degree      : 2

→  When ready, run Cell 6 to start training.


### ⚙️ Cell 7: Apply Source Fixes (run before training)

This cell patches three issues that cause training to crash at iteration 0:

1. **In-place tensor accumulation in `renderer.py`** — PyTorch's autograd cannot backprop through `canvas += ...` or `accum_alpha *= ...` because those ops mutate a tensor that is already part of the computation graph.  The fix rewrites every `+=` and `*=` in the renderer to their out-of-place equivalents using a tokeniser-safe regex.
2. **Missing `torch.cuda.empty_cache()` in training loop** — without periodic cache flushing the T4's 15.6 GB fills up and causes OOM around iter 3000+. The fix inserts a flush every 500 iters.
3. **Config caps** — ensures `config.yaml` on disk reflects the memory-safe values used at runtime (5 k iters, 75 k Gaussians, SH degree 1) so a bare `load_config` call never silently loads larger values.

**Run this cell once after Cell 4 and before Cell 8.**


In [ ]:
# ── Cell 7: Apply Source Fixes ──────────────────────────────────────
import os, re, torch

SEP = "=" * 60

# ── 1. Fix config.yaml ───────────────────────────────────────────────────────
print(SEP)
print("1 / 3  Fixing config/config.yaml")
print(SEP)
config_path = "config/config.yaml"
with open(config_path) as f:
    cfg_txt = f.read()
orig = cfg_txt

replacements = [
    (r"iterations:\s*\d+",          "iterations: 5000"),
    (r"max_gaussians:\s*\d+",       "max_gaussians: 75000"),
    (r"sh_degree:\s*\d+",           "sh_degree: 1"),
    (r"densify_until_iter:\s*\d+",  "densify_until_iter: 4000"),
    (r"opacity_reset_interval:\s*\d+", "opacity_reset_interval: 2000"),
]
for pattern, repl in replacements:
    cfg_txt = re.sub(pattern, repl, cfg_txt)

if cfg_txt != orig:
    with open(config_path, "w") as f:
        f.write(cfg_txt)
    print("  ✓ config.yaml updated (iterations=5k, max_gaussians=75k, sh_degree=1)")
else:
    print("  ✓ config.yaml already correct — no changes needed.")

# ── 2. Fix renderer.py (in-place → out-of-place) ────────────────────────────
print()
print(SEP)
print("2 / 3  Fixing src/renderer/renderer.py (in-place ops)")
print(SEP)
renderer_path = "src/renderer/renderer.py"
with open(renderer_path) as f:
    rend = f.read()
orig_rend = rend

# Replace ALL `lhs += rhs` → `lhs = lhs + (rhs)` and `lhs *= rhs` → `lhs = lhs * (rhs)`
# Matches any identifier (with optional indexing) followed by += or *=
# Uses a word-boundary-safe pattern; skips comment lines.
def fix_inplace(text):
    lines = text.split("\n")
    out = []
    changed = 0
    for line in lines:
        stripped = line.lstrip()
        if stripped.startswith("#"):
            out.append(line)
            continue
        # += fix
        m = re.match(r"(^(\s*))(\w+(?:\[[^\]]+\])?)\s*\+=(.*)", line)
        if m:
            indent, var, rhs = m.group(2), m.group(3), m.group(4)
            line = f"{indent}{var} = {var} + ({rhs.strip()})"
            changed += 1
        # *= fix
        m = re.match(r"(^(\s*))(\w+(?:\[[^\]]+\])?)\s*\*=(.*)", line)
        if m:
            indent, var, rhs = m.group(2), m.group(3), m.group(4)
            line = f"{indent}{var} = {var} * ({rhs.strip()})"
            changed += 1
        out.append(line)
    return "\n".join(out), changed

rend_fixed, n_changed = fix_inplace(rend)
if n_changed > 0:
    with open(renderer_path, "w") as f:
        f.write(rend_fixed)
    print(f"  ✓ Replaced {n_changed} in-place operation(s) in renderer.py")
else:
    print("  ✓ No in-place operations found in renderer.py — already clean.")

# ── 3. Fix trainer.py (periodic cache clearing) ──────────────────────────────
print()
print(SEP)
print("3 / 3  Fixing src/reconstruction/trainer.py (cache clearing)")
print(SEP)
trainer_path = "src/reconstruction/trainer.py"
with open(trainer_path) as f:
    trainer = f.read()

cache_snippet = (
    'if self.device == "cuda" and it % 500 == 0:\n'
    '                    torch.cuda.empty_cache()'
)
if cache_snippet not in trainer:
    # Insert after pbar.set_postfix(...) line
    trainer_fixed = re.sub(
        r"(^[ \t]*pbar\.set_postfix\(.+\)[ \t]*)$",
        r"\1\n                if self.device == \"cuda\" and it % 500 == 0:\n"
        r"                    torch.cuda.empty_cache()",
        trainer,
        flags=re.MULTILINE,
    )
    if trainer_fixed != trainer:
        with open(trainer_path, "w") as f:
            f.write(trainer_fixed)
        print("  ✓ Inserted cache-clearing block into trainer.py")
    else:
        print("  ⚠  Could not find pbar.set_postfix — cache clearing NOT inserted.")
        print("     Open trainer.py and manually add after the progress-bar update:")
        print('     if self.device == "cuda" and it % 500 == 0: torch.cuda.empty_cache()')
else:
    print("  ✓ Cache clearing already present in trainer.py — no changes needed.")

# ── Final GPU cache clear ─────────────────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

print()
print(SEP)
print("✅  ALL FIXES APPLIED — proceed to Cell 8 to train.")
print(SEP)


In [ ]:
# ── Cell 8: Train Gaussian Splat model ──────────────────────────────
# Estimated runtime on free Colab T4: ~7 min (5 k iters) to ~40 min (30 k iters).
# The config is capped in Cell 7. Adjust there if you want a faster preview run.
import sys, os, numpy as np, torch
from PIL import Image
from pathlib import Path

# Env must be set BEFORE importing torch.cuda
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

sys.path.insert(0, "/content/monosplat")

from src.utils.config_loader import load_config
from src.preprocessing.utils import load_colmap_model
from src.reconstruction.gaussian_model import GaussianModel
from src.reconstruction.trainer import GaussianTrainer
from src.renderer.camera import Camera as ViewerCamera
from src.renderer.renderer import GaussianRenderer

# ── Patch 1: robust _knn_mean_dist ───────────────────────────────────────────
def _knn_mean_dist_robust(pts: torch.Tensor, k: int = 3):
    N = pts.shape[0]
    if N == 0:
        return torch.tensor([], device=pts.device)
    k_actual = min(k, N - 1)
    if k_actual == 0:
        return torch.zeros(N, device=pts.device)
    dist_matrix = torch.cdist(pts, pts)
    dist_matrix.fill_diagonal_(float("inf"))
    knn_dists = torch.topk(dist_matrix, k=k_actual, largest=False, dim=1).values
    return knn_dists.mean(dim=1).clamp(min=1e-7)

from src.reconstruction import gaussian_model as _gm
if hasattr(_gm, "_knn_mean_dist"):
    _gm._knn_mean_dist = _knn_mean_dist_robust
    print("✅  Patched _knn_mean_dist")

# ── Patch 2: wrap densify / prune ops under no_grad ──────────────────────────
import functools

def _no_grad_wrap(fn):
    @functools.wraps(fn)
    def _inner(*a, **kw):
        with torch.no_grad():
            return fn(*a, **kw)
    return _inner

for _method in ("reset_opacity", "densify_and_prune", "prune_points"):
    if hasattr(GaussianModel, _method):
        setattr(GaussianModel, _method, _no_grad_wrap(getattr(GaussianModel, _method)))
        print(f"✅  Patched GaussianModel.{_method}")

# ── Load config ───────────────────────────────────────────────────────────────
cfg = load_config("config/config.yaml")

# Hard caps — never exceed free-tier T4 limits regardless of config.yaml values
cfg.training.iterations    = min(cfg.training.iterations,    5_000)
cfg.renderer.max_gaussians = min(cfg.renderer.max_gaussians, 75_000)
cfg.renderer.sh_degree     = min(cfg.renderer.sh_degree,     1)
cfg.viewer.window_width    = min(cfg.viewer.window_width,    640)
cfg.viewer.window_height   = min(cfg.viewer.window_height,   360)

print(f"\nConfig: iterations={cfg.training.iterations:,}, "
      f"max_gaussians={cfg.renderer.max_gaussians:,}, "
      f"sh_degree={cfg.renderer.sh_degree}, "
      f"resolution={cfg.viewer.window_width}x{cfg.viewer.window_height}")

# ── Load COLMAP ───────────────────────────────────────────────────────────────
cameras_colmap, images_colmap, points3d = load_colmap_model(COLMAP_DIR)

# ── Build training camera + image list ───────────────────────────────────────
W, H = cfg.viewer.window_width, cfg.viewer.window_height
train_cameras, train_images = [], []

for img_data in images_colmap.values():
    cam_data = cameras_colmap[img_data.camera_id]
    cam      = ViewerCamera.from_colmap(img_data, cam_data, W, H)
    img_path = Path(FRAMES_DIR) / img_data.name
    if not img_path.exists():
        img_path = Path(FRAMES_DIR) / Path(img_data.name).name
    if img_path.exists():
        img = (
            np.array(Image.open(img_path).convert("RGB").resize((W, H), Image.LANCZOS),
                     dtype=np.float32) / 255.0
        )
        train_cameras.append(cam)
        # Keep on CPU — loaded to GPU per iteration inside trainer
        train_images.append(torch.from_numpy(img).permute(2, 0, 1).cpu())
    else:
        print(f"⚠️  Missing frame: {img_data.name} — skipping")

print(f"Loaded {len(train_cameras)} cameras, {len(points3d):,} 3D points")
if not train_cameras:
    raise RuntimeError("No training images loaded. Check FRAMES_DIR and COLMAP_DIR paths.")

# ── Initialise Gaussian model from COLMAP point cloud ────────────────────────
xyzs = np.array([p.xyz for p in points3d.values()], dtype=np.float32)
rgbs = np.array([p.rgb for p in points3d.values()], dtype=np.float32) / 255.0

model = GaussianModel(sh_degree=cfg.renderer.sh_degree)
model.create_from_points(xyzs, rgbs)

# ── Create renderer + trainer ─────────────────────────────────────────────────
torch.cuda.empty_cache()
renderer = GaussianRenderer(width=W, height=H, device="cuda", batch_size=5_000)
trainer  = GaussianTrainer(model, renderer.render_torch, train_cameras, train_images, cfg)

# ── Train ─────────────────────────────────────────────────────────────────────
print(f"\nStarting training for {cfg.training.iterations:,} iterations...")
trainer.train()
print("\n✅  Training complete!")


In [ ]:
# ── Cell 9: Download output files ───────────────────────────────────
# Downloads <job_id>.splat and <job_id>.ply to your computer.
# The .splat is what the browser viewer uses.
# The .ply can be opened in SuperSplat (https://supersplat.playcanvas.com).

import glob, os, shutil
from google.colab import files

outputs = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, '*.ply')) +
    glob.glob(os.path.join(OUTPUT_DIR, '*.splat')),
    key=os.path.getmtime
)

if not outputs:
    print('❌  No output files found. Make sure Cell 6 completed successfully.')
else:
    downloaded = []
    for ext in ['.splat', '.ply']:
        matching = [f for f in outputs if f.endswith(ext)]
        if matching:
            latest = matching[-1]
            dest   = os.path.join(OUTPUT_DIR, f'{JOB_ID}{ext}')
            if os.path.abspath(latest) != os.path.abspath(dest):
                shutil.copy2(latest, dest)
            size_mb = os.path.getsize(dest) / 1e6
            print(f'  Downloading {JOB_ID}{ext}  ({size_mb:.1f} MB)...')
            files.download(dest)
            downloaded.append(f'{JOB_ID}{ext}')

    print(f'\n✅  Downloaded: {", ".join(downloaded)}')
    print()
    print('─' * 60)
    print('NEXT STEPS on your desktop:')
    print('─' * 60)
    print(f'  1. Create this folder if it does not exist:')
    print(f'       work\\{JOB_ID}\\models\\gaussian\\')
    print()
    print(f'  2. Move the downloaded files there:')
    print(f'       {JOB_ID}.splat  →  work\\{JOB_ID}\\models\\gaussian\\')
    print(f'       {JOB_ID}.ply    →  work\\{JOB_ID}\\models\\gaussian\\')
    print()
    print(f'  3. Open models/registry.json and find the entry for job {JOB_ID}.')
    print(f'     Update these three fields:')
    print(f'       "status":     "ready",')
    print(f'       "splat_path": "work/{JOB_ID}/models/gaussian/{JOB_ID}.splat",')
    print(f'       "ply_path":   "work/{JOB_ID}/models/gaussian/{JOB_ID}.ply"')
    print()
    print(f'  4. Make sure your local server is running:')
    print(f'       uvicorn src.pipeline.server:app --reload --port 8000')
    print()
    print(f'  5. Open in your browser:')
    print(f'       http://localhost:8000/viewer/{JOB_ID}')
    print()
    print('  Or drag the .splat directly to: https://supersplat.playcanvas.com')